# PubMed-Research-Agent · Hello-Agents 毕业设计 Demo

> 输入一个研究问题（如 "SEC61G in Lung Cancer"），系统自动完成 **PubMed 检索 → 大模型文献总结** 的完整链路，输出结构化研究报告。

- **作者**：@0609x
- **框架**：HelloAgents（`SimpleAgent` + `ToolRegistry` + `HelloAgentsLLM`）
- **模型**：OpenAI 兼容接口（DeepSeek / Qwen / GPT），默认复用 `.env` 中的 DeepSeek 配置

## 0. 环境准备

```bash
pip install -r requirements.txt
cp .env.example .env   # 然后编辑 .env，至少填写 LLM_API_KEY
```

In [ ]:
# ========================================
# 第 1 部分：环境配置
# ========================================

# 安装依赖（如已安装可跳过）
# !pip install -q -r requirements.txt

In [ ]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# 项目根目录（Notebook 在项目根目录运行）
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")


def _resolve(key: str, default: str = "") -> str:
    return os.getenv(key) or default


# 将项目 .env 中的 LLM 配置映射为 HelloAgents 环境变量
base_url = _resolve("LLM_BASE_URL", _resolve("LLM_API_BASE", "https://api.deepseek.com/v1"))
if "deepseek.com" in base_url and not base_url.rstrip("/").endswith("/v1"):
    base_url = base_url.rstrip("/") + "/v1"

os.environ["LLM_BASE_URL"] = base_url
os.environ.setdefault("LLM_MODEL_ID", _resolve("LLM_MODEL", "deepseek-chat"))
os.environ.setdefault("LLM_API_KEY", _resolve("LLM_API_KEY", ""))
os.environ.setdefault("LLM_TIMEOUT", _resolve("LLM_TIMEOUT", "60"))

assert os.environ.get("LLM_API_KEY"), "请在 .env 中配置 LLM_API_KEY（或 LLM_API_KEY）"

print("LLM 配置：")
print("  model   :", os.environ["LLM_MODEL_ID"])
print("  base_url:", os.environ["LLM_BASE_URL"])

## 第 2 部分：定义 HelloAgents 工具

将项目的两个核心能力封装为 HelloAgents `Tool` 子类：

1. **PubMedSearchAgentTool**：调用 NCBI E-utilities 检索 PubMed，返回结构化文献信息
2. **LiteratureSummaryAgentTool**：调用大模型对多篇摘要生成 5 维度结构化综述

In [ ]:
# ========================================
# 第 2.1 部分：PubMed 检索工具
# ========================================

from typing import Any, Dict, List

from hello_agents.tools import Tool, ToolParameter

from backend.tools.pubmed_tool import PubMedSearchTool as _PubMedSearchTool

PUBMED_EMAIL = _resolve("PUBMED_EMAIL", "pubmed.research.agent@example.com")
PUBMED_API_KEY = _resolve("PUBMED_API_KEY") or None
PUBMED_VERIFY_SSL = _resolve("PUBMED_VERIFY_SSL", "true").lower() != "false"


class PubMedSearchAgentTool(Tool):
    """HelloAgents 工具：PubMed 文献检索（NCBI E-utilities）。"""

    def __init__(self):
        super().__init__(
            name="pubmed_search",
            description="检索 PubMed 文献库，返回结构化文献信息（标题/摘要/PMID/DOI/作者/期刊/发表日期）。",
        )
        self._tool = _PubMedSearchTool(
            email=PUBMED_EMAIL,
            api_key=PUBMED_API_KEY,
            verify_ssl=PUBMED_VERIFY_SSL,
        )

    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="query",
                type="string",
                description="研究问题或检索关键词，例如：SEC61G in Lung Cancer",
                required=True,
            ),
            ToolParameter(
                name="max_results",
                type="integer",
                description="最多返回的文献数量，默认 5",
                required=False,
                default=5,
            ),
        ]

    def run(self, parameters: Dict[str, Any]) -> str:
        query = str(parameters.get("query", "")).strip()
        max_results = int(parameters.get("max_results") or 5)
        if not query:
            return json.dumps({"error": "query 不能为空"}, ensure_ascii=False)

        result = self._tool.search(query, max_results=max_results)
        articles = []
        for art in result.articles:
            d = art.to_dict()
            # 控制上下文长度：摘要截断到 1200 字符
            if len(d.get("abstract", "")) > 1200:
                d["abstract"] = d["abstract"][:1200] + "..."
            articles.append(d)

        return json.dumps(
            {"query": query, "total_count": result.total_count, "articles": articles},
            ensure_ascii=False,
        )

In [ ]:
# ========================================
# 第 2.2 部分：文献总结工具
# ========================================

from backend.services.literature_summary import LiteratureSummarizer


class LiteratureSummaryAgentTool(Tool):
    """HelloAgents 工具：基于 PubMed 摘要生成结构化文献综述。"""

    def __init__(self):
        super().__init__(
            name="literature_summary",
            description="基于多个 PubMed 文献摘要，调用大模型生成结构化综述（研究背景/当前研究热点/主要发现/实验验证方法/未来研究方向）。",
        )
        self._summarizer = LiteratureSummarizer(
            api_base=os.environ["LLM_BASE_URL"],
            api_key=os.environ["LLM_API_KEY"],
            model=os.environ["LLM_MODEL_ID"],
            temperature=0.3,
            timeout=float(_resolve("LLM_TIMEOUT", "240")),
        )

    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="articles",
                type="string",
                description="PubMed 检索结果的 JSON 数组字符串（由 pubmed_search 工具输出）",
                required=True,
            ),
            ToolParameter(
                name="language",
                type="string",
                description="输出语言：en 或 zh，默认 zh",
                required=False,
                default="zh",
            ),
        ]

    def run(self, parameters: Dict[str, Any]) -> str:
        raw = parameters.get("articles")
        if isinstance(raw, str):
            articles = json.loads(raw)
        else:
            articles = raw
        language = str(parameters.get("language") or "zh")
        summary = self._summarizer.summarize(articles, language=language)
        return json.dumps(summary.model_dump(), ensure_ascii=False, indent=2)

## 第 3 部分：直接调用链路（Part A）

先直接调用两个工具，验证检索与总结能力（不依赖 LLM 的工具调用格式，最稳定）。

In [ ]:
# ========================================
# 第 3 部分：直接调用两个工具
# ========================================

QUERY = "SEC61G in Lung Cancer"   # 可修改为任意研究问题
MAX_RESULTS = 5

search_tool = PubMedSearchAgentTool()
summary_tool = LiteratureSummaryAgentTool()

print("=" * 60)
print("Step 1: PubMed 检索 ->", QUERY)
print("=" * 60)
search_data = json.loads(search_tool.run({"query": QUERY, "max_results": MAX_RESULTS}))
articles = search_data.get("articles", [])
print(f"PubMed 命中: {search_data.get('total_count', 0)} 篇，展示前 {len(articles)} 篇")
for i, art in enumerate(articles, 1):
    print(f"  [{i}] PMID:{art['pmid']} | {art['title'][:70]} | {art.get('journal', '')[:30]} | {art.get('publish_date', '')}")

print()
print("=" * 60)
print("Step 2: 大模型文献总结（language=zh）")
print("=" * 60)
summary = json.loads(
    summary_tool.run({"articles": json.dumps(articles, ensure_ascii=False), "language": "zh"})
)
print("研究背景:", str(summary.get("research_background", ""))[:180], "...")
print("当前研究热点:")
for h in summary.get("current_hotspots", []):
    print("  -", h.get("topic", ""), ":", str(h.get("description", ""))[:100])
print("主要发现:")
for f in summary.get("main_findings", [])[:3]:
    print("  -", str(f)[:140])
print("未来研究方向:")
for d in summary.get("future_directions", [])[:3]:
    print("  -", d.get("direction", ""), ":", str(d.get("rationale", ""))[:80])

## 第 4 部分：SimpleAgent 智能体调用（Part B）

使用 HelloAgents 的 `SimpleAgent` + `ToolRegistry`，让大模型自动规划并依次调用两个工具，生成完整研究报告。

In [ ]:
# ========================================
# 第 4 部分：构建并运行 HelloAgents 智能体
# ========================================

from hello_agents import HelloAgentsLLM, SimpleAgent, ToolRegistry

# 注册工具
registry = ToolRegistry()
registry.register_tool(search_tool)
registry.register_tool(summary_tool)

# 创建 LLM（显式指定 custom provider，使用 .env 配置）
llm = HelloAgentsLLM(
    model=os.environ["LLM_MODEL_ID"],
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"],
    provider="custom",
    temperature=0.3,
)

system_prompt = """你是一名科研文献分析助手。用户会提出研究问题，请按以下流程执行：
1. 使用 pubmed_search 工具检索 PubMed 文献（参数：query=研究问题, max_results=5）
2. 使用 literature_summary 工具对检索结果进行总结（参数：articles=检索结果JSON, language=zh）
3. 基于工具返回结果，以 Markdown 输出最终报告，包含：研究背景 / 当前研究热点 / 主要发现 / 实验验证方法 / 未来研究方向

工具调用格式示例：
[TOOL_CALL:pubmed_search:query=SEC61G in Lung Cancer,max_results=5]
[TOOL_CALL:literature_summary:articles={"articles":[...]},language=zh]
"""

agent = SimpleAgent(
    name="科研文献助手",
    llm=llm,
    system_prompt=system_prompt,
    tool_registry=registry,
)

print("=== 智能体运行中（检索+总结，通常需要 30-120 秒）===")
agent_result = agent.run("请检索 SEC61G 在肺癌中的研究文献，并总结研究热点与未来方向")
print(agent_result)

In [ ]:
# ========================================
# 第 5 部分：保存研究报告
# ========================================

report_md = f"""# 研究报告

## 查询
{QUERY}

## 检索概览
- 检索模式：advanced（高级检索）
- PubMed 命中数：{search_data.get('total_count', 0)} 篇
- 展示文献：{len(articles)} 篇

## 文献列表
"""
for i, art in enumerate(articles, 1):
    report_md += (
        f"{i}. **{art.get('title', '')}**\n"
        f"   - PMID: {art.get('pmid', '')} | DOI: {art.get('doi', '') or '-'} | "
        f"{art.get('journal', '')} | {art.get('publish_date', '')}\n"
    )

report_md += "\n## AI 总结（直接调用链路）\n\n" + json.dumps(summary, ensure_ascii=False, indent=2)
report_md += "\n\n## 智能体报告（SimpleAgent）\n\n" + agent_result

output_path = ROOT / "outputs" / "sample_report.md"
output_path.write_text(report_md, encoding="utf-8")
print("报告已保存:", output_path)

## 项目总结

### 实现的功能

- PubMed 自动检索（NCBI E-utilities），返回标题/摘要/PMID/DOI/作者/期刊/发表日期
- 大模型 5 维度结构化文献总结（研究背景/研究热点/主要发现/实验方法/未来方向）
- HelloAgents 工具系统封装 + `SimpleAgent` 智能体自动规划调用

### 遇到并解决的挑战

- NCBI 限流：默认 3 req/s，内置节流；配置 API Key 后提升至 10 req/s
- 上下文长度：摘要截断策略控制输入 token
- 多模型兼容：统一 OpenAI 兼容接口，DeepSeek / Qwen / GPT 一键切换

### 未来改进方向

- 接入 RAG（Qdrant 语义检索 + RRF 融合）与知识图谱（Neo4j）
- 支持更多检索源（bioRxiv / Google Scholar）
- 文献引用格式导出（BibTeX / RIS）

> 感谢 Datawhale 社区与 Hello-Agents 项目！